In [ ]:
email = "polina.teryaninova@multonpartners.com"
folder_path = r"C:\Users\kitit\Desktop\выгрузка\3806"
sender = ""
mail_date = "2026-11-16"


In [ ]:
%pip install -r "C:/Users/kitit/Desktop/выгрузка/requirements.txt"

In [ ]:
import datetime
import os
import zipfile
import win32com.client as win32


In [ ]:
def outlook_inbox(namespace, mailbox):
    wanted = mailbox.strip().lower()
    if wanted:
        accounts = namespace.Accounts
        for i in range(1, accounts.Count + 1):
            account = accounts.Item(i)
            smtp = (account.SmtpAddress or "").strip().lower()
            if smtp == wanted:
                return account.DeliveryStore.GetDefaultFolder(6)
        stores = namespace.Stores
        for i in range(1, stores.Count + 1):
            store = stores.Item(i)
            display = (store.DisplayName or "").strip().lower()
            if wanted in display:
                return store.GetDefaultFolder(6)
    return namespace.GetDefaultFolder(6)


def message_sender(message):
    try:
        if message.SenderEmailType == "EX":
            user = message.Sender.GetExchangeUser()
            if user is not None:
                return user.PrimarySmtpAddress
    except Exception:
        pass
    return message.SenderEmailAddress or ""


def parse_day(value):
    if isinstance(value, datetime.datetime):
        return value.date()
    if isinstance(value, datetime.date):
        return value
    return datetime.datetime.strptime(str(value).strip(), "%Y-%m-%d").date()


def safe_name(name):
    bad = '<>:"/\\|?*'
    cleaned = "".join("_" if ch in bad else ch for ch in name).strip().rstrip(".")
    return cleaned or "attachment"


def save_message_attachment(message, subject, dest_dir):
    """Распаковывает zip и называет файл темой письма."""
    archives = []
    for i in range(1, message.Attachments.Count + 1):
        attachment = message.Attachments.Item(i)
        if os.path.splitext(attachment.FileName)[1].lower() == ".zip":
            archives.append(attachment)

    if not archives:
        print(f"В письме нет zip: {subject}")
        return False

    subject_name = safe_name(subject)
    print(f"{subject}: письмо от {message.ReceivedTime}, архивов {len(archives)}")

    for index, attachment in enumerate(archives, start=1):
        suffix = "" if len(archives) == 1 else f"_{index}"
        archive_path = os.path.join(dest_dir, f"{subject_name}{suffix}.zip")
        attachment.SaveAsFile(archive_path)
        with zipfile.ZipFile(archive_path) as zf:
            members = [n for n in zf.namelist() if n and not n.endswith("/")]
            if not members:
                print(f"Пустой архив: {subject}")
            for member in members:
                inner_name = os.path.basename(member.replace("\\", "/"))
                inner_ext = os.path.splitext(inner_name)[1]
                if len(members) == 1:
                    target_name = f"{subject_name}{suffix}{inner_ext}"
                else:
                    target_name = f"{subject_name}{suffix}_{safe_name(inner_name)}"
                target = os.path.join(dest_dir, target_name)
                with zf.open(member) as src, open(target, "wb") as dst:
                    dst.write(src.read())
                print(f"Распаковано: {target}")
        os.remove(archive_path)

    return True


def save_archives_for_day(inbox, dest_dir, sender, day_value):
    wanted = sender.strip().lower()
    if not wanted:
        print("Не задан адрес отправителя")
        return

    day = parse_day(day_value)
    os.makedirs(dest_dir, exist_ok=True)
    start = datetime.datetime.combine(day, datetime.time.min)
    end = start + datetime.timedelta(days=1)
    items = inbox.Items
    items.Sort("[ReceivedTime]", True)
    prop = "urn:schemas:httpmail:datereceived"
    start_text = start.strftime("%Y-%m-%d %H:%M")
    end_text = end.strftime("%Y-%m-%d %H:%M")
    restriction = f'@SQL="{prop}" >= \'{start_text}\' AND "{prop}" < \'{end_text}\''
    try:
        found = items.Restrict(restriction)
    except Exception:
        found = items

    seen = set()
    saved = 0
    message = found.GetFirst()
    while message is not None:
        try:
            received = message.ReceivedTime
            received_day = datetime.date(received.year, received.month, received.day)
            subject = (message.Subject or "").strip()
            if (
                received_day == day
                and message_sender(message).strip().lower() == wanted
                and subject
                and subject not in seen
            ):
                seen.add(subject)
                if save_message_attachment(message, subject, dest_dir):
                    saved += 1
        except Exception as e:
            print(f"Ошибка письма: {e}")
        message = found.GetNext()

    print(f"Сохранено писем: {saved}")


In [ ]:
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
inbox = outlook_inbox(namespace, email)
save_archives_for_day(inbox, folder_path, sender, mail_date)
